In [9]:
import pandas as pd
import numpy as np
import urllib.parse
from sqlalchemy import create_engine, text

# DB 정보
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "dbname": "postgres",
    "user": "postgres",
    "password": "leejangwoo1!",
}

def get_engine():
    p = urllib.parse.quote_plus(DB_CONFIG["password"])
    conn_str = f"postgresql+psycopg2://{DB_CONFIG['user']}:{p}@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['dbname']}"
    return create_engine(conn_str, future=True)

engine = get_engine()

# FCT1~4 조회
fct_tables = ["FCT1_machine_log", "FCT2_machine_log", "FCT3_machine_log", "FCT4_machine_log"]
df_list = []

with engine.connect() as conn:
    for tbl in fct_tables:
        sql = text(f"""
            SELECT end_day, station, dayornight, contents, time
            FROM d1_machine_log."{tbl}"
            WHERE contents LIKE 'Barcode Scan ::%'
        """)
        df_tmp = pd.read_sql(sql, conn)
        df_list.append(df_tmp)

df = pd.concat(df_list, ignore_index=True)

# 바코드 추출
df["barcode"] = df["contents"].str.split("::").str[-1].str.strip()

# PD / Non-PD
df["remark"] = df["barcode"].apply(lambda x: "PD" if isinstance(x, str) and len(x)>=18 and x[17] in ("J","S") else "Non-PD")

# 정렬
df = df.sort_values(["end_day", "dayornight", "time"]).reset_index(drop=True)

# end_ts 생성
df["time_clean"] = df["time"].astype(str).str.split(".").str[0]
df["end_ts"] = pd.to_datetime(df["end_day"].astype(str) + " " + df["time_clean"],
                              format="%Y%m%d %H:%M:%S", errors="coerce")

# CT 계산
df["ct"] = df.groupby("station")["end_ts"].diff().dt.total_seconds()

df_box = df.dropna(subset=["ct"]).copy()

print("[INFO] df_box ready:", df_box.shape)

[INFO] df_box ready: (91945, 10)


In [10]:
clean = df_box.copy()

# (1) CT > 600 제거
clean = clean[clean["ct"] <= 600]

# (2) shift 경계 제거
clean["end_time"] = clean["end_ts"].dt.time

clean = clean[
    ~(
        clean["end_time"].between(pd.to_datetime("08:20:00").time(),
                                  pd.to_datetime("08:30:00").time())
        |
        clean["end_time"].between(pd.to_datetime("20:20:00").time(),
                                  pd.to_datetime("20:30:00").time())
    )
]

# (3) PD → Non-PD 전환 시 첫 1개 제거
clean = clean.sort_values(["station", "end_ts"]).reset_index(drop=True)
remove_idx = []

for st, st_df in clean.groupby("station", observed=True):
    st_df = st_df.reset_index()
    for i in range(1, len(st_df)):
        if st_df.loc[i-1, "remark"] == "PD" and st_df.loc[i, "remark"] == "Non-PD":
            remove_idx.append(st_df.loc[i, "index"])

clean = clean[~clean.index.isin(remove_idx)]

# (4) remark = PD & CT ≥ 37.1 제거
clean = clean[~((clean["remark"]=="PD") & (clean["ct"] >= 37.1))]

# (5) remark = Non-PD & CT ≥ 31.9 제거
clean = clean[~((clean["remark"]=="Non-PD") & (clean["ct"] >= 31.9))]

clean = clean.reset_index(drop=True)

print("[INFO] clean CT size:", clean.shape)

[INFO] clean CT size: (24673, 11)


In [12]:
rows = []
group_cols = ["end_day", "station", "dayornight", "remark"]

for keys, group in clean.groupby(group_cols, observed=True):
    ct_vals = group["ct"].dropna()
    sample_size = len(ct_vals)

    # 샘플 수가 30 이하인 그룹은 제외
    if sample_size <= 30:
        continue

    median_val = ct_vals.median()

    rows.append({
        "end_day": keys[0],
        "station": keys[1],
        "dayornight": keys[2],
        "remark": keys[3],
        "ct_median": median_val,
        "sample_size": sample_size
    })

ct_median_df = (
    pd.DataFrame(rows)
    .sort_values(group_cols)
    .reset_index(drop=True)
)

display(ct_median_df)

,end_day,station,dayornight,remark,ct_median,sample_size
0,20250930,FCT2,night,PD,36.0,361
1,20251001,FCT1,day,PD,36.0,63
2,20251001,FCT1,night,PD,37.0,121
3,20251001,FCT2,day,PD,36.0,175
4,20251001,FCT2,night,PD,37.0,103
...,...,...,...,...,...,...
147,20251118,FCT3,day,Non-PD,31.0,62
148,20251118,FCT3,night,Non-PD,31.0,113
149,20251118,FCT4,night,Non-PD,31.0,92
150,20251119,FCT3,night,PD,37.0,49


In [13]:
import urllib.parse
from sqlalchemy import create_engine

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "dbname": "postgres",
    "user": "postgres",
    "password": "leejangwoo1!",
}

def get_engine():
    pwd = urllib.parse.quote_plus(DB_CONFIG["password"])
    conn_str = f"postgresql+psycopg2://{DB_CONFIG['user']}:{pwd}@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['dbname']}"
    engine = create_engine(conn_str, future=True)
    return engine

engine = get_engine()
print("[INFO] DB Engine Ready")

[INFO] DB Engine Ready


In [14]:
from sqlalchemy import text

schema_name = "d1_machine_log"
table_name = "FCT_operation_CT"

# 테이블 재생성 (기존 있으면 삭제)
with engine.begin() as conn:
    conn.execute(text(f"""
        DROP TABLE IF EXISTS {schema_name}.{table_name} CASCADE;
        
        CREATE TABLE {schema_name}.{table_name} (
            end_day        VARCHAR(8),
            station        VARCHAR(10),
            dayornight     VARCHAR(10),
            remark         VARCHAR(10),
            ct_median      DOUBLE PRECISION,
            sample_size    INTEGER
        );
    """))

print("[INFO] New table created:", f"{schema_name}.{table_name}")

# DataFrame → PostgreSQL 저장
ct_median_df.to_sql(
    name=table_name,
    con=engine,
    schema=schema_name,
    if_exists="append",
    index=False
)

print("[INFO] Data inserted successfully!")

[INFO] New table created: d1_machine_log.FCT_operation_CT
[INFO] Data inserted successfully!
